# P1.01 · Monitoring: stability and performance
**End-to-end credit decision engine** · **L3** tier

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/1thirteeng3/ds-ai-cookbook/blob/colab/en/projects/P1-credit/P1.01-e2e-credit-decision-engine/notebooks/34-monitoring.ipynb)

| Level | Colab runtime | Time (smoke / full) | Document |
|---|---|---|---|
| 🟠 L3 | cpu | 1 / 4 min | [index.en.md](../index.en.md) |

> Score PSI, feature CSI and performance under five controlled scenarios, including two data failures. The goal is to learn to read the alarms: stability is not performance.

## 0. Parameters

`RUN_MODE="smoke"` uses 10% of the data (or the synthetic twin with `OFFLINE=True`) and runs in minutes. `RUN_MODE="full"` uses the whole dataset and produces the official results. `OFFLINE=True` exists only for tests and CI: never draw conclusions from it.

In [ ]:
# Parameters (papermill overrides this cell)
RUN_MODE = "smoke"   # "smoke" | "full"
SEED = 42
DATA_DIR = "data"    # Colab + Drive: "/content/drive/MyDrive/cookbook-data"
OFFLINE = False      # True: synthetic twin (tests/CI). Never for full results.
SAMPLE_FRAC = 0.1    # smoke only
N_BOOT = 200         # bootstrap resamples for metric uncertainty

## 1. Setup

On Colab, installs the `cookbook` package from the private repo (`GH_TOKEN` secret) and the exact versions used for the results. NumPy, pandas and scikit-learn keep Colab's versions and are recorded in the result file. If the install upgrades any of them, use *Runtime > Restart session* and run from section 0. To download Home Credit, create the `KAGGLE_API_TOKEN` secret (or `KAGGLE_USERNAME` and `KAGGLE_KEY`) and accept the competition rules on kaggle.com with the same account.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = {
    "optbinning": "1.0.0",
    "lightgbm": "4.7.0",
    "optuna": "5.0.0",
    "mlflow": "3.18.0",
    "fastapi": "0.162.0",
    "httpx": "0.28.1",
    "polars": "2.1.0",
    "pyarrow": "26.0.1",
    "kaggle": "2.2.5",
    "joblib": "1.6.0",
    "matplotlib": "3.11.2",
}

if IN_COLAB and not sys.modules.get("cookbook"):
    from google.colab import userdata

    token = userdata.get("GH_TOKEN")
    pins = " ".join(f"{k}=={v}" for k, v in PACKAGES.items())
    %pip install -q uv
    !uv pip install --system -q {pins} "cookbook[credit,kaggle] @ git+https://{token}@github.com/1thirteeng3/cookbokk.git@main"

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from cookbook import credit as cr
from cookbook import home_credit as hc
from cookbook.data import sample
from cookbook.env import colab_secrets_to_env, hardware_info, quiet_logs, set_seeds, versions
from cookbook.results import Timer, save_result

quiet_logs()  # before heavy imports: silences optional-solver probing
import json

import joblib
import lightgbm as lgb

set_seeds(SEED)
colab_secrets_to_env()
if RUN_MODE == "full" and OFFLINE:
    raise ValueError("RUN_MODE='full' needs real data (OFFLINE=False).")

SOURCE = "home-credit-synthetic" if OFFLINE else "home-credit"
ARTIFACTS = Path(DATA_DIR) / "artifacts" / "P1.01" / RUN_MODE
ARTIFACTS.mkdir(parents=True, exist_ok=True)
HW = hardware_info()
VERSIONS = versions([*PACKAGES, "numpy", "pandas", "scikit-learn"])
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.width", 140)
print(SOURCE, RUN_MODE, ARTIFACTS, HW)

In [ ]:
missing = [f for f in ["features.parquet", "feature_meta.json", "lgbm.txt", "calibrator.joblib"] if not (ARTIFACTS / f).exists()]
if missing:
    raise FileNotFoundError(
        f"Missing {missing} in {ARTIFACTS}. Run notebooks 30 and 31 first "
        f"with the same RUN_MODE and DATA_DIR. / Rode 30 e 31 antes."
    )

## 2. Reference and scenarios

**Reference:** training data. **Current:** the test set, first unchanged and then under **simulated** scenarios (the data has no time dimension, so we create the shifts in a controlled way):

| Scenario | What changes | Type |
|---|---|---|
| no change | nothing | control |
| income +25% | nominal income rises, credit does not | economy |
| younger population | under-30s weighted ×3 | client mix |
| EXT_SOURCE_3 outage | 40% arrive missing | data quality |
| bureau feed down | half the clients without history | data quality |

In [ ]:
n_twin = int(307_511 * (SAMPLE_FRAC if RUN_MODE == "smoke" else 1))
app, bureau, DATASET = hc.load(SOURCE, DATA_DIR, n_synthetic=n_twin, seed=SEED)
if RUN_MODE == "smoke" and not OFFLINE:
    app = sample(app, SAMPLE_FRAC, SEED, stratify=hc.TARGET)
    bureau = bureau[bureau[hc.ID].isin(app[hc.ID])]
DATASET["sample_frac"] = SAMPLE_FRAC if RUN_MODE == "smoke" else 1.0
DATASET["n_rows"] = len(app)
print(app.shape, bureau.shape, f"default rate = {app[hc.TARGET].mean():.4f}")

In [ ]:
meta = json.loads((ARTIFACTS / "feature_meta.json").read_text())
COLS = meta["model_columns"]
booster = lgb.Booster(model_file=str(ARTIFACTS / "lgbm.txt"))
calibrator = joblib.load(ARTIFACTS / "calibrator.joblib")

def score(feats):
    X, _ = hc.to_categorical(feats[COLS], meta["categories"])
    return calibrator.predict(booster.predict(X))

feats_all = pd.read_parquet(ARTIFACTS / "features.parquet")
reference = feats_all[feats_all["SPLIT"] == "train"]
ref_pd = score(reference)

test_ids = feats_all.loc[feats_all["SPLIT"] == "test", hc.ID]
app_test = app[app[hc.ID].isin(test_ids)].reset_index(drop=True)
bureau_test = bureau[bureau[hc.ID].isin(test_ids)]
rng = np.random.default_rng(SEED)

def scenario(name):
    a, b = app_test.copy(), bureau_test.copy()
    if name == "income +25%":
        a["AMT_INCOME_TOTAL"] *= 1.25
    elif name == "younger population":
        w = np.where(-a["DAYS_BIRTH"] / 365.25 < 30, 3.0, 1.0)
        picked = a.sample(n=len(a), replace=True, weights=w, random_state=SEED)
        old_ids = picked[hc.ID].to_numpy()
        a = picked.reset_index(drop=True)
        a[hc.ID] = np.arange(len(a)) + 10_000_000      # resampled rows get new ids...
        mapping = pd.DataFrame({"old": old_ids, hc.ID: a[hc.ID]})
        b = (b.merge(mapping, left_on=hc.ID, right_on="old", suffixes=("_old", ""))
              .drop(columns=["old", f"{hc.ID}_old"]))   # ...and keep their own bureau history
    elif name == "EXT_SOURCE_3 outage":
        a.loc[rng.random(len(a)) < 0.4, "EXT_SOURCE_3"] = np.nan
    elif name == "bureau feed down":
        dropped = a[hc.ID].sample(frac=0.5, random_state=SEED)
        b = b[~b[hc.ID].isin(dropped)]
    return a, b

## 3. Solution

For each scenario: we recompute features and PDs through the same pipeline, measure the PD's PSI against the reference, the features' CSI, and (because the target is available here) Gini and calibration-in-the-large: mean predicted PD against observed default rate.

In [ ]:
SCENARIOS = ["no change", "income +25%", "younger population", "EXT_SOURCE_3 outage", "bureau feed down"]
rows, top_csi = [], {}
for name in SCENARIOS:
    a, b = scenario(name)
    feats = hc.build_features(a, b)
    p = score(feats)
    csi = cr.csi(reference[COLS], feats[COLS])
    top_csi[name] = csi.head(3).round(3).to_dict()
    rows.append({
        "scenario": name,
        "psi_pd": cr.psi(ref_pd, p),
        "max_csi": csi.iloc[0],
        "gini": cr.gini(feats[hc.TARGET], p),
        "mean_pd": p.mean(),
        "default_rate": feats[hc.TARGET].mean(),
    })
MONITOR = pd.DataFrame(rows).set_index("scenario")
MONITOR["psi_alert"] = MONITOR["psi_pd"].map(lambda v: cr.psi_label(v, "en"))
MONITOR["csi_alert"] = MONITOR["max_csi"].map(lambda v: cr.psi_label(v, "en"))
MONITOR

In [ ]:
pd.Series(top_csi, name="top-3 CSI features").to_frame()

## 4. Validation: how to read the table

Read the columns in pairs, always against the **no change** row:

- **`psi_pd` × `gini`:** the PD's PSI measures whether the risk distribution shifted; Gini measures whether the model still ranks. One can move without the other.
- **`max_csi`:** points to the feature that shifted most. In data failures (EXT_SOURCE_3, bureau) it is the clearest alarm, even when the PD's PSI barely moves, because the model partly compensates with other variables.
- **`mean_pd` × `default_rate`:** calibration-in-the-large. If they drift apart, PDs no longer work as probabilities and notebook 32's policy goes wrong, even with Gini intact.
- **Rule of thumb (PSI and CSI):** < 0.10 stable · 0.10–0.25 attention · > 0.25 significant shift. These are industry conventions, not statistical tests (see section 5).

## 5. Variation axes

### 5.1 Number of PSI bins

PSI depends on the number of bins. With too many bins, sampling noise looks like "shift".

In [ ]:
p_now = score(hc.build_features(app_test, bureau_test))
pd.Series({bins: cr.psi(ref_pd, p_now, bins=bins) for bins in (5, 10, 20, 50, 100)}, name="PSI, no change")

## 6. Pitfalls demonstrated

### 6.1 Treating a stability alarm as a performance drop

In the income +25% scenario, the income CSI flags a shift while Gini stays practically the same. Retraining just because of the alarm may make the model worse; check performance and calibration first.

### 6.2 Monitoring only the score

In data failures, the PD's PSI tends to move little, because the model compensates with other variables, yet Gini drops and a whole source is broken. Without CSI and per-source null and volume checks, the incident goes unnoticed.

## 7. Results

In [ ]:
DATASET["n_rows"] = len(app_test)
for name, r in MONITOR.iterrows():
    save_result(
        recipe_id="P1.01", notebook="34-monitoring", stack="psi-csi", variant=name, run_mode=RUN_MODE,
        hardware=HW, versions=VERSIONS, dataset=DATASET, params={"bins": 10},
        metrics={k: {"mean": round(float(r[k]), 6), "std": None, "n": 1}
                 for k in ("psi_pd", "max_csi", "gini", "mean_pd", "default_rate")},
        time_s={"train": 0.0, "inference_per_1k": None},
    )
MONITOR.round(4)

## 8. Conclusion

- PSI and CSI tell **what** changed; performance and calibration tell **whether it matters**. Both together, plus per-source quality checks.
- In production: the same computation daily/weekly, with alerts and delayed labels (recipes G3.02, G3.04, N1.13).
- **L4 tier:** [40 · reject inference](40-reject-inference.ipynb) and [41 · tabular deep learning](41-tabular-deep-learning.ipynb).